# Figure 2 — cell coverage across z

Number of detected cells per z-slice in DMSO-treated spheroids, for both cell lines.
Establishes that segmentation recovers cells throughout the spheroid depth rather
than only at the surface.

**Ported from** `spher_colo52_v1/3_Figure2/CellCoverage/3_CellCoverage.ipynb`.
Two changes were required:

1. The original read `FeaturesImages_100125_none`, which no longer exists anywhere.
   Extraction stamps are re-runs of the same images (see
   `provenance/PORT_TRIAGE.md`), so this uses `011225` from the main tree.
2. `100125` used an older column convention. Translated here:
   `Cytoplasm_ObjectNumber` → `ObjectNumber_cytoplasm`,
   `Nuclei_ObjectNumber` → `ObjectNumber_nuclei`,
   `Cytoplasm_AreaShape_Area` → `AreaShape_Area_cytoplasm`,
   `Metadata_cmpd_cmpdname` → `Metadata_cmpdname`,
   `Metadata_cmpd_cell_line` → `Metadata_cell_line`.

The original's `savefig` was commented out, so the archived PDF was saved by hand.
Here `save_panel` writes the figure and its source table together.

In [ ]:
# Path bootstrap — works at any folder depth, no chdir, no absolute paths
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from utils.paths import features, require
from utils.panels import save_panel

In [ ]:
# Which feature extraction to read. All exp1 stamps are re-runs of the same
# images; 011225 lives in the main tree, so using it avoids depending on the fork.
FEATURE_VERSION = "011225"
CELL_LINES = ["HT29", "HCT116"]
TRT_TYPE = "dmso"

# Columns needed for cell counting, in the current naming convention.
GROUP_COLS = ["Metadata_PlateWell", "Metadata_Site", "Metadata_cell_line"]
COUNT_COLS = ["ObjectNumber_cytoplasm", "ObjectNumber_nuclei"]
AREA_COL = "AreaShape_Area_cytoplasm"
KEEP = GROUP_COLS + COUNT_COLS + [AREA_COL, "Metadata_cmpdname"]

In [ ]:
# Scan lazily and push the DMSO filter and column projection into the read: the
# single-cell parquets are ~4.8 GB (HCT116) and ~2.7 GB (HT29), and only 7 of
# their 2167 columns are needed here.
frames = []
for line in CELL_LINES:
    path = require(features("exp1_main", FEATURE_VERSION, "SingleCell", f"{line}.parquet"))
    lf = (
        pl.scan_parquet(path)
        .filter(pl.col("Metadata_cmpdname") == TRT_TYPE)
        .select(KEEP)
    )
    frames.append(lf)
    print(f"{line}: scanning {path.name}")

df = pl.concat(frames).collect()
print(f"\nDMSO single cells: {df.height:,} rows")
print(f"z-slices present: {sorted(df['Metadata_Site'].unique().to_list())}")
print(f"cell lines: {df['Metadata_cell_line'].unique().to_list()}")

In [ ]:
def calc_numcells(df: pl.DataFrame) -> pl.DataFrame:
    """Collapse single cells to one row per (well, z-slice, cell line).

    Cell count per slice is the MAX object number in that slice (objects are
    numbered sequentially within an image), and area is summed across objects.
    """
    return (
        df.with_columns(pl.col("ObjectNumber_cytoplasm").cast(pl.Float64))
        .group_by(GROUP_COLS)
        .agg(
            [pl.max(c).alias(c) for c in COUNT_COLS]
            + [pl.sum(AREA_COL).alias(AREA_COL)]
        )
        .sort("Metadata_Site")
    )


per_slice = calc_numcells(df)
print(f"{per_slice.height} (well x z-slice x cell line) rows")
per_slice.head()

In [ ]:
data = per_slice.to_pandas()

fig, ax = plt.subplots(figsize=(6, 6))
sns.boxplot(
    data=data, x="Metadata_Site", y="ObjectNumber_cytoplasm",
    width=0.5, fill=True, hue="Metadata_cell_line", palette="dark:grey",
    fliersize=1, showfliers=False, legend=False, ax=ax,
)
ax.set_xlabel("Z-slice", fontsize=12)
ax.set_ylabel("Number of cells", fontsize=12)
ax.set_ylim(-0.4, 210)

save_panel(
    fig, "Fig2d", data=data,
    caption="Cells per spheroid per z-slice, DMSO wells, both cell lines",
    notebook="analysis/3_Figure2/CellCoverage/3_CellCoverage.ipynb",
)

## Sanity check: expected cells per spheroid from geometry

Independent order-of-magnitude check on the counts above — how many cells a
spheroid of this diameter should contain, from the per-cell area observed in a
single slice.

In [ ]:
number_of_cells = 60          # typical objects in one mid-spheroid slice
pixel_area = 210_000          # spheroid cross-sectional area, px
resolution = 0.227            # um per px
diameter_spheroid = 130       # um

area_per_cell = pixel_area / number_of_cells
diameter = 2 * (area_per_cell / 3.141592653589793) ** 0.5 * resolution
volume = (4 / 3) * 3.141592653589793 * (diameter / 2) ** 3
volume_spheroid = (4 / 3) * 3.141592653589793 * (diameter_spheroid / 2) ** 3

print(f"diameter of a cell:            {diameter:.2f} um")
print(f"expected cells per spheroid:   {volume_spheroid / volume:.0f}")